In [85]:
import pandas as pd
import numpy as np

In [86]:
df = pd.read_csv(r'C:\Users\Junayed\pandas_prac\Aug_3\Car_Rental\messy_carrental_day3b.csv')

In [87]:
df.head(5)

,ReservationID,CustomerName,Email,Phone,CarType,PickupDate,ReturnDate,RentalDays,DailyRate,TotalCost,InsuranceOpted,DriverLicenseNo,Notes,Status
0,R001,Owen Marsh,owen.marsh@gmail.com,312-555-0111,Economy,2023-05-01,2023-05-04,3.0,$45.00,$165.00,Yes,456781,NaN,Completed
1,R002,Bianca Ruiz,bianca.ruiz@gmail.com,312-555-0122,economy,05/02/2023,05/05/2023,3.0,45,$135.00,No,456782,NaN,completed
2,R003,Femi Adeyemi,femi.adeyemi@gmail.com,312.555.0133,SUV,05-03-2023,NaN,NaN,$89.00,NaN,Pending,456783,Return date TBD,Active
3,R004,Katarina Novak,katarina.novak@hotmail.com,NaN,Compact,2023-05-05,2023-05-08,3.0,$55.00,$195.00,Yes,456784,,Completed
4,R005,Owen Marsh,owen.marsh@gmail.com,312-555-0111,Economy,05/01/2023,05/04/2023,3.0,$45.00,$165.00,Yes,456781,NaN,Completed


In [88]:
df.shape

(28, 14)

In [89]:
df.dtypes

ReservationID          str
CustomerName           str
Email                  str
Phone                  str
CarType                str
PickupDate             str
ReturnDate             str
RentalDays         float64
DailyRate              str
TotalCost              str
InsuranceOpted         str
DriverLicenseNo      int64
Notes                  str
Status                 str
dtype: object

- The 'CarType' is not well formatted, we need to capitalize the types. 

In [90]:
df["CarType"] = df["CarType"].astype(str).str.strip().str.title()
df["CarType"]

0     Economy
1     Economy
2         Suv
3     Compact
4     Economy
5     Econimy
6         Suv
7     Compact
8     Economy
9         Suv
10    Economy
11        Suv
12    Compact
13    Economy
14    Economy
15        Suv
16    Econimy
17    Compact
18        Suv
19        Suv
20    Economy
21    Compact
22    Economy
23        Suv
24    Economy
25        Suv
26    Compact
27    Economy
Name: CarType, dtype: str

In [91]:
df["CarType"].value_counts()

CarType
Economy    11
Suv         9
Compact     6
Econimy     2
Name: count, dtype: int64

- the dates both on PickupDate and ReturnDate unformatted and messy, (e.g. 04-06-2023, 2023-04-05, 04/01/2023 ). With to_datetime and format='mixed' we can fix the date style to a clear format Y-MM-DD.

In [119]:
df["PickupDate"] = pd.to_datetime(df["PickupDate"].astype(str).str.strip(), format='mixed', errors='coerce')
df["PickupDate"] = df["PickupDate"].dt.strftime('%Y-%m-%d')

In [120]:
df["ReturnDate"] = pd.to_datetime(df["ReturnDate"].astype(str).str.strip(), format='mixed',errors='coerce')
df["ReturnDate"] = df["ReturnDate"].dt.strftime('%Y-%m-%d')

- We already have a RentedDays column, we can't belive this until we verify if the returndate and pickupdate giving us the accorate renting days, otherwise we will get wrong price and days. Let's calculate the days rented.

In [94]:
df["RentalDays_cal"] = (df["ReturnDate"] - df["PickupDate"]).dt.days
df["RentalDays_cal"]

0     3.0
1     3.0
2     NaN
3     3.0
4     3.0
5     3.0
6     3.0
7     1.0
8     3.0
9     3.0
10    1.0
11    NaN
12    3.0
13    2.0
14    1.0
15    3.0
16    3.0
17    1.0
18    3.0
19    3.0
20    1.0
21    3.0
22    3.0
23    1.0
24    3.0
25    3.0
26    3.0
27    1.0
Name: RentalDays_cal, dtype: float64

- We have calculated the rented days and now let's check if their is any mismatch with the existing renteddays column, if we find any then we have to input the correct duration.

In [95]:
df[["PickupDate", "ReturnDate", "RentalDays", "RentalDays_cal"]]

,PickupDate,ReturnDate,RentalDays,RentalDays_cal
0,2023-05-01,2023-05-04,3.0,3.0
1,2023-05-02,2023-05-05,3.0,3.0
2,2023-05-03,NaT,NaN,NaN
3,2023-05-05,2023-05-08,3.0,3.0
4,2023-05-01,2023-05-04,3.0,3.0
5,2023-05-06,2023-05-09,3.0,3.0
6,2023-05-06,2023-05-09,3.0,3.0
7,2023-05-07,2023-05-08,1.0,1.0
8,2023-05-07,2023-05-10,3.0,3.0
9,2023-05-06,2023-05-09,3.0,3.0


- We can see a mismatch on index 13 and few NaN value which we have to check why that happend. On index 11, we have the input of how many days it is been rented so we can we add the date after 3 days of pickup date and on index 19 we do not have have any rental days data but we have clear pickup and rental date so we can calculate this.

In [96]:
mismatch_rentaldate = (df["RentalDays_cal"] < 0) | (df["RentalDays"] != df["RentalDays_cal"])

df.loc[mismatch_rentaldate, ["PickupDate", "ReturnDate", "RentalDays", "RentalDays_cal"]]

,PickupDate,ReturnDate,RentalDays,RentalDays_cal
2,2023-05-03,NaT,NaN,NaN
11,2023-05-09,NaT,3.0,NaN
13,2023-05-10,2023-05-12,5.0,2.0
19,2023-05-13,2023-05-16,NaN,3.0


2. We do not have any return date nor any rentaldays data, so we can't touch it.
11. We have rentaldays but not the returndate, we can add the date of after 3 days of pickup on this.
13. The car was rented for 2 days but it is written as 5 days, we have to fix that.
19. We have both dates but NaN at rentaldays, we have to fix that too.

In [97]:
calc_days = df["PickupDate"] + pd.to_timedelta(df["RentalDays"], unit='D')
df["ReturnDate"] = df["ReturnDate"].fillna(calc_days)

df["RentalDays"] = df["RentalDays_cal"].fillna(df["RentalDays"])

In [98]:
df[["PickupDate", "ReturnDate", "RentalDays", "RentalDays_cal"]]

,PickupDate,ReturnDate,RentalDays,RentalDays_cal
0,2023-05-01,2023-05-04,3.0,3.0
1,2023-05-02,2023-05-05,3.0,3.0
2,2023-05-03,NaT,NaN,NaN
3,2023-05-05,2023-05-08,3.0,3.0
4,2023-05-01,2023-05-04,3.0,3.0
5,2023-05-06,2023-05-09,3.0,3.0
6,2023-05-06,2023-05-09,3.0,3.0
7,2023-05-07,2023-05-08,1.0,1.0
8,2023-05-07,2023-05-10,3.0,3.0
9,2023-05-06,2023-05-09,3.0,3.0


- From the rental rate we have remove the 'dollar' sign and convert to numeric so that we can do calculation later 

In [101]:
df["DailyRate"] = pd.to_numeric(df["DailyRate"].astype(str).str.replace("$","",regex=False).str.replace(",","",regex=False), errors='coerce')

df["TotalCost"] = pd.to_numeric(df["TotalCost"].astype(str).str.replace("$","",regex=False).str.replace(",","",regex=False), errors='coerce')

df["TotalCost_cal"] = (df["DailyRate"] * df["RentalDays"])

df[["RentalDays","DailyRate", "TotalCost", "TotalCost_cal", "InsuranceOpted"]]

,RentalDays,DailyRate,TotalCost,TotalCost_cal,InsuranceOpted
0,3.0,45.0,165.0,135.0,Yes
1,3.0,45.0,135.0,135.0,No
2,NaN,89.0,NaN,NaN,Pending
3,3.0,55.0,195.0,165.0,Yes
4,3.0,45.0,165.0,135.0,Yes
5,3.0,45.0,135.0,135.0,No
6,3.0,89.0,297.0,267.0,Yes
7,1.0,55.0,55.0,55.0,Pending
8,3.0,45.0,135.0,135.0,No
9,3.0,89.0,297.0,267.0,Yes


As we can see there is mismatch but the difference is most probably the insurance fee, 
how? if we look at the chart, we can see all the 'Yes' rows have mismatched with our 
calculation, so we can consider it as insurance fee. We can test a hypothesis.

Checked the actual mismatch amount instead of just guessing - it's not random, it's 
exactly $10 per day for every 'Yes' row (e.g. row 0: 165-135=30, and RentalDays=3, so 
30/3=10 per day). Same $10/day shows up on every other Yes row too, so that's not a 
coincidence.

Also checked the 'Pending' rows to see if they get charged too - turns out they match 
the plain rate*days calc with no extra fee, so looks like Pending = not charged (yet), 
same as No. Going with that assumption since the numbers back it up consistently.

In [104]:
insurance_fee = np.where(df["InsuranceOpted"].str.lower() == "yes", 10* df["RentalDays"], 0)

df["TotalCost_calc_with_ins"] = df["RentalDays"] * df["DailyRate"] + insurance_fee

price_mismatch = (df["TotalCost"] - df["TotalCost_calc_with_ins"]).abs() > 0.01

df.loc[price_mismatch, ["ReservationID", "InsuranceOpted", "DailyRate", "RentalDays", "TotalCost", "TotalCost_calc_with_ins"]]

,ReservationID,InsuranceOpted,DailyRate,RentalDays,TotalCost,TotalCost_calc_with_ins


Yeah, the extra price was for insurance.

In [108]:
df["InsuranceOpted"].value_counts()

InsuranceOpted
No         12
Yes        11
Pending     5
Name: count, dtype: int64

In [107]:
df["Notes"] = df["Notes"].astype(str).str.strip().str.title()
df["Notes"] = df["Notes"].replace(["", "nan", "None"], np.nan)

In [106]:
df["Status"] = df["Status"].astype(str).str.strip().str.title()

Lastly, we have to remove the duplicates.

In [109]:
dup_cols = ["CustomerName", "Email", "CarType", "PickupDate"]

df[df.duplicated(subset=dup_cols, keep = False)]

,ReservationID,CustomerName,Email,Phone,CarType,PickupDate,ReturnDate,RentalDays,DailyRate,TotalCost,InsuranceOpted,DriverLicenseNo,Notes,Status,RentalDays_cal,TotalCost_cal,Insurance_cal,TotalCost_calc_with_ins
0,R001,Owen Marsh,owen.marsh@gmail.com,312-555-0111,Economy,2023-05-01,2023-05-04,3.0,45.0,165.0,Yes,456781,NaN,Completed,3.0,135.0,165.0,165.0
1,R002,Bianca Ruiz,bianca.ruiz@gmail.com,312-555-0122,Economy,2023-05-02,2023-05-05,3.0,45.0,135.0,No,456782,NaN,Completed,3.0,135.0,135.0,135.0
4,R005,Owen Marsh,owen.marsh@gmail.com,312-555-0111,Economy,2023-05-01,2023-05-04,3.0,45.0,165.0,Yes,456781,NaN,Completed,3.0,135.0,165.0,165.0
6,R007,Priya Kapoor,priya.kapoor@gmail.com,312-555-0155,Suv,2023-05-06,2023-05-09,3.0,89.0,297.0,Yes,456787,NaN,Active,3.0,267.0,297.0,297.0
9,R010,Priya Kapoor,priya.kapoor@gmail.com,312-555-0155,Suv,2023-05-06,2023-05-09,3.0,89.0,297.0,Yes,456787,Possible Duplicate,Active,3.0,267.0,297.0,297.0
24,R025,Bianca Ruiz,bianca.ruiz@gmail.com,312-555-0122,Economy,2023-05-02,2023-05-05,3.0,45.0,135.0,No,456782,Duplicate?,Completed,3.0,135.0,135.0,135.0


We have 3 duplicate entry, let's remove them.

In [110]:
df = df.drop_duplicates(subset=dup_cols, keep='first').reset_index(drop=True)

In [111]:
df.dtypes

ReservationID                         str
CustomerName                          str
Email                                 str
Phone                                 str
CarType                               str
PickupDate                 datetime64[us]
ReturnDate                 datetime64[us]
RentalDays                        float64
DailyRate                         float64
TotalCost                         float64
InsuranceOpted                        str
DriverLicenseNo                     int64
Notes                                 str
Status                                str
RentalDays_cal                    float64
TotalCost_cal                     float64
Insurance_cal                     float64
TotalCost_calc_with_ins           float64
dtype: object

In [114]:
df = df.drop(columns = ["RentalDays_cal", "TotalCost_cal", "Insurance_cal", "TotalCost_calc_with_ins"])

In [115]:
df.dtypes

ReservationID                 str
CustomerName                  str
Email                         str
Phone                         str
CarType                       str
PickupDate         datetime64[us]
ReturnDate         datetime64[us]
RentalDays                float64
DailyRate                 float64
TotalCost                 float64
InsuranceOpted                str
DriverLicenseNo             int64
Notes                         str
Status                        str
dtype: object

In [121]:
df.to_excel("Clean_Rental_Car_data.xlsx", index=False)